# Sentiment Analysis of Customer Service Conversations

**Course:** DI 725 - Transformers and Attention-Based Deep Networks  
**Student Number:** 2786028  
**Model:** RoBERTa-base (fine-tuning)  
**Task:** 3-class sentiment classification (positive, negative, neutral)

This notebook implements a complete pipeline for sentiment analysis on customer service conversations using a fine-tuned RoBERTa model. Key design decisions:

- **Metadata prepend:** Categorical features (issue area, complexity, product, agent level) are prepended as text to the conversation
- **Head+Tail truncation:** First 128 + last 382 tokens to capture both problem statement and resolution
- **Class-weighted loss:** To handle severe class imbalance (positive class ~1.7%)
- **Experiment tracking:** All experiments logged to Weights & Biases

## 1. Setup & Imports

In [ ]:
!pip install transformers wandb scikit-learn -q

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import RobertaTokenizer, RobertaForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)

import wandb
import warnings
warnings.filterwarnings('ignore')

# --- Reproducibility ---
SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(SEED)

# --- Device ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
wandb.login()

## 2. Data Loading

In [ ]:
# Update this path if running on Google Colab
DATA_DIR = 'Assignment - 1 Dataset'

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f"Train set: {train_df.shape}")
print(f"Test set:  {test_df.shape}")
print(f"\nColumns: {list(train_df.columns)}")
train_df.head(2)